# مشروع بيان — خط الأنابيب المترابط
# Bayan — End-to-End Pipeline

**البرنامج | Programme:** SDA-AIE-211 — Natural Language Processing with Transformers
**إعداد وتقديم المدربة | Instructor:** ميعاد المري · Meaad Al-Marri
**المتدربة | Student:** غالا الشريف · Ghala Alshreef
**المستودع | Repository:** https://github.com/lolo67fa/bayan-nlp-ghalaalshreef-sdaia

---

## ما هذا الدفتر؟ | What is this notebook?

هذا الدفتر يجمع مخرجات مختبرات الأيام الأربعة في **خط أنابيب واحد متصل**: كل مرحلة تستهلك مخرجات المرحلة التي تسبقها فعليًا، من النص الخام إلى خدمة مختبرة.

This notebook links the four days into **one connected pipeline** — each stage consumes the previous stage's output, from raw text to a tested service.

دفاتر الأيام الأصلية (`00` إلى `08`) تبقى في `notebooks/` بوصفها أدلة التشغيل اليومية. هذا الدفتر لا يحل محلها؛ بل يعرض المشروع مترابطًا.

The original day notebooks remain in `notebooks/` as per-day evidence. This notebook does not replace them; it presents the project as one connected system.

---

## المسار | The pipeline

| # | المرحلة | الناتج |
|---|---|---|
| 0 | الإعداد | بيئة مثبتة، بذرة واحدة، جهاز معلن |
| 1 | البيانات والتقسيم | `group_overlap == 0` |
| 2 | المعالجة | عقد النسختين + PII + profile عربية معلنة |
| 3 | الترميز | fertility + truncation |
| 4 | المهام | تصنيف + NER + QA |
| 5 | البحث الدلالي | FAISS + threshold مجمّد |
| 6 | التقييم | CI + شرائح + تصنيف أخطاء |
| 7 | التحسين والخدمة | ONNX + parity + FastAPI |
| 8 | الملخص | كل رقم مع وسمه ومصدره |

---

## قاعدة الصدق | Honesty rule

كل رقم في هذا الدفتر يحمل وسمًا. لا رقم بلا وسم.

| الوسم | المعنى |
|---|---|
| `MEASURED_SMOKE` | مقاس على عينة الدورة الصغيرة. يثبت سلامة المسار، لا جاهزية الإنتاج. |
| `COURSE_FIXTURE` | تنبؤات تقدمها الدورة لتعليم منهج التقييم. ليست مخرجات نماذج المشروع. |
| `MEASURED` / `PROJECT_ARTIFACT` | مقاس على مصنوعة المشروع الفعلية. |

---
# 0) الإعداد | Setup

النسخ مثبتة لأن الرقم الذي لا يمكن إعادة إنتاجه لا قيمة له.

Versions are pinned because a number you cannot reproduce is not a result.

In [ ]:
import importlib.metadata, importlib.util, subprocess, sys

PINNED = {
    "transformers": "5.15.1",
    "tokenizers": "0.22.2",
    "spacy": "3.8.7",
    "scikit-learn": "1.9.0",
}
missing = []
for dist, expected in PINNED.items():
    try:
        if importlib.metadata.version(dist) != expected:
            missing.append(f"{dist}=={expected}")
    except importlib.metadata.PackageNotFoundError:
        missing.append(f"{dist}=={expected}")
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

print("Python:", sys.version.split()[0])
print("Pinned:", {d: importlib.metadata.version(d) for d in PINNED})
print("SETUP=PASS")

In [ ]:
import csv, html, io, json, math, os, random, re, unicodedata, urllib.request
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
except ImportError:
    torch = None
    DEVICE = "cpu"

REPORTS = Path("reports"); REPORTS.mkdir(exist_ok=True)
COURSE_RAW = "https://raw.githubusercontent.com/almiyead-rgb/bayan-applied-nlp-course/main"

def fetch_rows(url, fallback, parser):
    """Fetch course data; fall back to the embedded sample when offline."""
    try:
        with urllib.request.urlopen(url, timeout=20) as response:
            return parser(response.read().decode("utf-8")), "github_course_file"
    except Exception as exc:
        return fallback, f"embedded_fallback:{type(exc).__name__}"

print("SEED", SEED)
print("DEVICE", DEVICE)
print("ENVIRONMENT=PASS")

---
# 1) البيانات والتقسيم | Data and splits

**السؤال:** هل يمكن لصياغتين من نفس الشكوى أن تتفرقا بين التدريب والاختبار؟

إذا حدث، النموذج **يتذكّر** ولا **يعمّم**، ودرجتك تطلع عالية وهي كذبة. `group_id` يربط النسخ المتشابهة ويمنع فصلها.

**The question:** can two phrasings of the same complaint straddle train and test? If so, the model memorises rather than generalises, and the score is a lie. `group_id` groups near-duplicates so they cannot be split.

In [ ]:
CLS_FALLBACK = [
    {"example_id": "F-001", "group_id": "DG-A", "split": "train", "language": "ar",
     "text": "تعذر تسجيل الدخول إلى البوابة", "topic": "digital_service", "sentiment": "negative"},
    {"example_id": "F-002", "group_id": "DG-A", "split": "train", "language": "en",
     "text": "I cannot sign in to the portal", "topic": "digital_service", "sentiment": "negative"},
    {"example_id": "F-003", "group_id": "DG-B", "split": "validation", "language": "ar",
     "text": "أحتاج معرفة حالة طلب التصريح", "topic": "permit", "sentiment": "neutral"},
    {"example_id": "F-004", "group_id": "DG-B", "split": "validation", "language": "en",
     "text": "I need the status of my permit request", "topic": "permit", "sentiment": "neutral"},
    {"example_id": "F-005", "group_id": "DG-C", "split": "test", "language": "ar",
     "text": "تأخر موعد العيادة هذا الصباح", "topic": "health", "sentiment": "negative"},
    {"example_id": "F-006", "group_id": "DG-C", "split": "test", "language": "en",
     "text": "My clinic appointment was delayed", "topic": "health", "sentiment": "negative"},
]

rows, DATA_SOURCE = fetch_rows(
    f"{COURSE_RAW}/data/sample/bayan_day2_classification.csv",
    CLS_FALLBACK,
    lambda text: list(csv.DictReader(io.StringIO(text))),
)

train_rows      = [r for r in rows if r["split"] == "train"]
validation_rows = [r for r in rows if r["split"] == "validation"]
test_rows       = [r for r in rows if r["split"] == "test"]
LABELS = sorted({r["topic"] for r in rows})

def validate_splits(train, validation, test):
    groups = [{r["group_id"] for r in part} for part in (train, validation, test)]
    overlap = (
        len(groups[0] & groups[1]) + len(groups[0] & groups[2]) + len(groups[1] & groups[2])
    )
    return {
        "group_overlap": overlap,
        "sizes": [len(train), len(validation), len(test)],
        "labels_per_split": [sorted({r["topic"] for r in p}) for p in (train, validation, test)],
    }

split_report = validate_splits(train_rows, validation_rows, test_rows)
print("DATA_SOURCE", DATA_SOURCE, "rows:", len(rows))
print("LABELS", LABELS)
print("SPLIT_REPORT", split_report)
assert split_report["group_overlap"] == 0, "Group leakage between splits"
print("SPLIT_ISOLATION=PASS")

---
# 2) المعالجة | Preprocessing

## عقد النسختين | The two-copy contract

`display_text` هو ما يراه المستخدم. `model_text` هو ما يدخل النموذج بعد الحماية والـ profile المعلنة.

**لماذا نسختان؟** في استخراج الإجابات، النموذج يجد الجواب عند موضع معيّن في نسخة النموذج. لو عرضتِ على نص مختلف، الإزاحات تنزاح والجواب يطلع مقصوصًا.

**Why two copies?** In extractive QA the model locates an answer at an offset in the model copy. Rendering against a different string shifts those offsets and truncates the answer.

## الـ profiles | Declared profiles

| Profile | التحويلات |
|---|---|
| `conservative` | NFC · إزالة التطويل · ضبط المسافات · إخفاء PII |
| `search` | ما سبق + إزالة التشكيل + توحيد الألف + `ى → ي` |

**التاء المربوطة `ة` لا تتحول إلى `ه` في أيٍّ منهما.** التحويل يدمج فروقًا ذات معنى وقد يفسد حدود الكيان في NER.

In [ ]:
ARABIC_DIACRITICS = re.compile(r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]")
TATWEEL   = "\u0640"
WHITESPACE = re.compile(r"\s+")
HTML_TAG   = re.compile(r"<[^>]+>")
EMAIL      = re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b")
SAUDI_MOBILE = re.compile(r"(?<!\d)(?:\+?966|00966|0)?5\d{8}(?!\d)")

PROFILE_VERSION = "1.0.0"
PROFILES = {
    "conservative": {"remove_diacritics": False, "normalize_alef": False, "normalize_ya": False},
    "search":       {"remove_diacritics": True,  "normalize_alef": True,  "normalize_ya": True},
}

@dataclass(frozen=True)
class TextRecord:
    display_text: str
    model_text: str
    profile: str
    profile_version: str

def mask_pii(text):
    return SAUDI_MOBILE.sub("<PHONE>", EMAIL.sub("<EMAIL>", text))

def prepare_text(text, profile="conservative"):
    if not isinstance(text, str):
        raise TypeError("text must be a string")
    options = PROFILES[profile]
    model = unicodedata.normalize("NFC", html.unescape(text))
    model = HTML_TAG.sub(" ", model).replace(TATWEEL, "")
    if options["remove_diacritics"]:
        model = ARABIC_DIACRITICS.sub("", model)
    if options["normalize_alef"]:
        model = re.sub(r"[إأآٱ]", "ا", model)
    if options["normalize_ya"]:
        model = model.replace("ى", "ي")
    model = WHITESPACE.sub(" ", mask_pii(model)).strip()
    return TextRecord(text, model, profile, PROFILE_VERSION)

# Golden tests run BEFORE any corpus is touched.
GOLDEN_CASES = [
    {"text": "إِدَارَةُ الحِساب",            "profile": "search",       "expected": "ادارة الحساب"},
    {"text": "إِدَارَةُ الحِساب",            "profile": "conservative", "expected": "إِدَارَةُ الحِساب"},
    {"text": "راسل test@example.com",     "profile": "search",       "expected": "راسل <EMAIL>"},
    {"text": "للتجربة فقط 0551234567",     "profile": "search",       "expected": "للتجربة فقط <PHONE>"},
]
for case in GOLDEN_CASES:
    actual = prepare_text(case["text"], case["profile"]).model_text
    status = "PASS" if actual == case["expected"] else "FAIL"
    print(status, repr(case["text"]), "->", repr(actual))
    assert actual == case["expected"], (case, actual)

# Taa marbuta must survive both profiles.
assert "ة" in prepare_text("الخدمة متأخرة", "search").model_text
print("GOLDEN_TESTS=PASS")

In [ ]:
ACTIVE_PROFILE = "conservative"   # D-001: no measurement yet justifies the stronger profile
records = [prepare_text(r["text"], ACTIVE_PROFILE) for r in rows]
model_texts = [rec.model_text for rec in records]

# The display copy must survive every transform.
assert all(rec.display_text == r["text"] for rec, r in zip(records, rows))
print("ACTIVE_PROFILE", ACTIVE_PROFILE, PROFILE_VERSION)
for rec in records[:4]:
    print(" ", repr(rec.display_text), "->", repr(rec.model_text))
print("TWO_COPY_CONTRACT=PASS")

---
# 3) الترميز | Tokenisation

**fertility** = عدد الـ tokens لكل كلمة. **truncation rate** = نسبة النصوص التي تتجاوز الحد فتُقص.

⚠️ **fertility أقل لا يعني tokenizer أفضل.** هو مؤشر تكلفة وخطر — طول أكبر، حوسبة أكثر، ضغط أبكر على حد السياق، ومحاذاة أصعب في NER. الحكم بالجودة يأتي من مقياس المهمة.

⚠️ الرموز الخاصة تُستبعد من الحساب. عدّها يضخّم الرقم على النصوص القصيرة.

In [ ]:
from transformers import AutoTokenizer

TOKENIZER_ID = "distilbert/distilbert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID, use_fast=True)
SPECIALS = set(tokenizer.all_special_tokens)

def word_count(text):
    return max(1, len(text.split()))

def token_fertility(text):
    tokens = [t for t in tokenizer.tokenize(text) if t not in SPECIALS]
    return len(tokens) / word_count(text)

def truncation_rate(texts, max_length):
    texts = list(texts)
    if not texts:
        raise ValueError("texts must not be empty")
    lengths = [len(tokenizer(t)["input_ids"]) for t in texts]
    return sum(length > max_length for length in lengths) / len(texts)

ar_texts = [rec.model_text for rec, r in zip(records, rows) if r["language"] == "ar"]
en_texts = [rec.model_text for rec, r in zip(records, rows) if r["language"] == "en"]

ar_fertility = float(np.mean([token_fertility(t) for t in ar_texts]))
en_fertility = float(np.mean([token_fertility(t) for t in en_texts]))

print(f"Arabic mean fertility  [MEASURED_SMOKE]: {ar_fertility:.2f}")
print(f"English mean fertility [MEASURED_SMOKE]: {en_fertility:.2f}")
for limit in (8, 16, 96):
    print(f"truncation @{limit} [MEASURED_SMOKE]: {truncation_rate(model_texts, limit):.0%}")

MAX_LENGTH = 96   # D-001: at 8 the truncation rate reaches 40% on the Day-1 sample
assert ar_fertility > 0 and en_fertility > 0
print("TOKENISATION_METRICS=PASS")

---
# 4) المهام | Task heads

مشفّر واحد، ثلاثة رؤوس. الفرق بين المهام هو شكل الـ labels ودالة الخسارة والمقياس — لا المشفّر.

## 4.1 خط الأساس قبل المحوّل | Baseline before transformer

بلا خط أساس، الرقم بلا مرجع. `char_wb` مقصود للعربية: يعمل على مستوى الحروف داخل حدود الكلمة، فيلتقط اللصائق (`و` + `ب` + `ال`) التي يفوّتها الترميز الكلمي.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score
from sklearn.pipeline import make_pipeline
from sklearn.svm import LinearSVC

baseline = make_pipeline(
    TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=1),
    LinearSVC(random_state=SEED),
)
baseline.fit([r["text"] for r in train_rows], [r["topic"] for r in train_rows])
baseline_predictions = baseline.predict([r["text"] for r in validation_rows])
baseline_f1 = f1_score(
    [r["topic"] for r in validation_rows], baseline_predictions,
    labels=LABELS, average="macro", zero_division=0,
)
print(f"Baseline validation macro-F1 [MEASURED_SMOKE]: {baseline_f1:.4f}")
assert 0.0 <= baseline_f1 <= 1.0
print("BASELINE=PASS")

## 4.2 محاذاة NER | NER alignment

الكلمة الواحدة تنكسر إلى عدة subwords، ولها label واحد. السياسة:

| النوع | Label |
|---|---|
| رموز خاصة | `-100` |
| أول subword | label الكلمة |
| باقي الـ subwords | `-100` |

`-100` تتجاهله دالة الخسارة. لو وضعتِ `O` بدلًا منه، النموذج يتعلّم أن نصف الكيان ليس كيانًا. المسألة أشد في العربية لأن اللصائق تكسر الكلمات أكثر.

**والقياس على مستوى الكيان لا الـ token:** كيان نصفه صحيح ليس كيانًا.

In [ ]:
def align_labels(words, tags):
    encoded = tokenizer(words, is_split_into_words=True, truncation=True, max_length=MAX_LENGTH)
    word_ids = encoded.word_ids()
    labels, previous = [], None
    for word_id in word_ids:
        if word_id is None:
            labels.append(-100)
        elif word_id != previous:
            labels.append(tags[word_id])
        else:
            labels.append(-100)
        previous = word_id
    return encoded, labels

words = ["تعطلت", "بوابة", "التصاريح", "في", "الرياض"]
tags  = ["O", "B-SERVICE", "I-SERVICE", "O", "B-LOCATION"]
encoded, aligned = align_labels(words, tags)
for token, label in zip(tokenizer.convert_ids_to_tokens(encoded["input_ids"]), aligned):
    print(f"  {token:<18} {label}")

assert aligned[0] == -100 and aligned[-1] == -100, "special tokens must be ignored"
assert sum(1 for x in aligned if x != -100) == len(words), "one supervised position per word"
print("NER_ALIGNMENT=PASS")

In [ ]:
def bio_entities(sequence):
    spans, start, label = [], None, None
    for index, tag in enumerate(list(sequence) + ["O"]):
        if tag.startswith("B-") or tag == "O" or (tag.startswith("I-") and label != tag[2:]):
            if start is not None:
                spans.append((label, start, index))
                start, label = None, None
        if tag.startswith("B-"):
            start, label = index, tag[2:]
    return spans

def entity_report(truths, predictions):
    gold = {(i, *s) for i, seq in enumerate(truths) for s in bio_entities(seq)}
    pred = {(i, *s) for i, seq in enumerate(predictions) for s in bio_entities(seq)}
    tp, fp, fn = len(gold & pred), len(pred - gold), len(gold - pred)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall    = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"precision": precision, "recall": recall, "f1": f1,
            "true_entities": len(gold), "predicted_entities": len(pred)}

# A partially matched entity scores zero, not half.
boundary = entity_report([["B-ORG", "I-ORG", "O"]], [["B-ORG", "O", "O"]])
print("Partial-span score:", boundary)
assert boundary["f1"] == 0.0, "entity scoring must be strict"
print("STRICT_ENTITY_BOUNDARY=PASS")

## 4.3 استخراج الإجابة والصدق في اللا-إجابة | Extractive QA and the honest null

نظام يستخرج جوابًا **دائمًا** هو نظام يهلوس. إذا لم يحتوِ السياق على إجابة، الجواب الصحيح هو `None`.

In [ ]:
def best_span(start_logits, end_logits, offsets, context,
              max_answer_length=48, top_k=20, null_threshold=0.0):
    starts = np.argsort(start_logits)[-top_k:][::-1]
    ends   = np.argsort(end_logits)[-top_k:][::-1]
    best = None
    for s in starts:
        for e in ends:
            if s > e or e - s + 1 > max_answer_length:
                continue
            if offsets[s] is None or offsets[e] is None:
                continue
            score = float(start_logits[s] + end_logits[e])
            if best is None or score > best["score"]:
                best = {"answer": context[offsets[s][0]:offsets[e][1]],
                        "score": score, "start": offsets[s][0], "end": offsets[e][1]}
    null_score = float(start_logits[0] + end_logits[0])
    if best is None:
        return {"answer": None, "reason": "no_candidate_span", "margin": null_score}
    margin = null_score - best["score"]
    if margin > null_threshold:
        return {"answer": None, "reason": "no_answer_in_context", "margin": margin}
    return {**best, "null_margin": margin}

context = "الخدمة متاحة في الرياض"
offsets = [None, (0, 6), (7, 12), (13, 15), (16, 22)]
answerable = best_span([0.0, 0.1, 0.1, 0.2, 4.0], [0.0, 0.1, 0.1, 0.2, 4.5], offsets, context)
unanswerable = best_span([5.0, 1.0, 2.0], [5.0, 1.0, 2.0],
                         [None, (0, 6), (7, 12)], "الخدمة متاحة")
print("Valid span:", answerable)
print("No answer :", unanswerable)
assert answerable["answer"] == "الرياض"
assert unanswerable["answer"] is None and unanswerable["reason"] == "no_answer_in_context"
print("QA_SPAN_AND_NULL=PASS")

---
# 5) البحث الدلالي | Semantic search

⚠️ **الفخ الأخطر:** الضرب الداخلي يساوي cosine **فقط** إذا كان الجانبان مطبَّعين إلى طول 1. تطبيع جانب واحد ينتج ترتيبًا يبدو معقولًا لكنه لم يعد يعني cosine — وهذا عطل لا يُكتشف بالنظر. لذلك نفحص المعيار على الجانبين.

**والحد يُضبط على validation فقط ثم يُجمَّد.** ضبط الحد على نفس البيانات التي تقيسين عليها ينتج رقمًا بلا معنى.

In [ ]:
def l2_normalise(matrix):
    matrix = np.asarray(matrix, dtype=np.float32)
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    return matrix / np.maximum(norms, 1e-12)

# Deterministic stand-in vectors keep this notebook runnable offline.
# The measured figures in §8 come from notebooks/06_semantic_search.ipynb.
corpus_vectors = l2_normalise(rng.normal(size=(len(rows), 16)))
query_vectors  = l2_normalise(rng.normal(size=(4, 16)))

assert np.allclose(np.linalg.norm(corpus_vectors, axis=1), 1.0, atol=1e-5)
assert np.allclose(np.linalg.norm(query_vectors,  axis=1), 1.0, atol=1e-5)

try:
    import faiss
    index = faiss.IndexFlatIP(corpus_vectors.shape[1])
    index.add(corpus_vectors)
    scores, ranked = index.search(query_vectors, min(3, len(rows)))
    backend = "faiss.IndexFlatIP"
except ImportError:
    similarity = query_vectors @ corpus_vectors.T
    ranked = np.argsort(-similarity, axis=1)[:, :3]
    scores = np.take_along_axis(similarity, ranked, axis=1)
    backend = "numpy inner product (faiss unavailable)"

FROZEN_THRESHOLD = 0.4592   # tuned on validation only in notebooks/06, then frozen

manifest = {
    "backend": backend,
    "vector_count": int(corpus_vectors.shape[0]),
    "embedding_dimension": int(corpus_vectors.shape[1]),
    "l2_normalised_both_sides": True,
    "frozen_threshold": FROZEN_THRESHOLD,
    "threshold_tuned_on": "validation_only",
    "reranking_decision": "REJECT_NO_MEASURED_LIFT",
}
print("SEARCH_MANIFEST", json.dumps(manifest, indent=2))
print("BOTH_SIDES_L2_NORMALISED=PASS")

---
# 6) التقييم | Evaluation

## لماذا فترة الثقة تغيّر القراءة

على 36 صفًا، نموذجان يفصل بينهما **0.0012** وفترتاهما متداخلتان بالكامل. الفارق ضجيج، لا تحسّن.

بلا فترة ثقة يُقرأ الرقم فوزًا. معها يُقرأ «لا نستطيع التمييز». وهذه القاعدة تحكم كل رقم سابق في هذا المشروع.

In [ ]:
def macro_f1(truth, prediction):
    labels = sorted(set(truth) | set(prediction))
    scores = []
    for label in labels:
        tp = sum(t == label and p == label for t, p in zip(truth, prediction))
        fp = sum(t != label and p == label for t, p in zip(truth, prediction))
        fn = sum(t == label and p != label for t, p in zip(truth, prediction))
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall    = tp / (tp + fn) if tp + fn else 0.0
        scores.append(2 * precision * recall / (precision + recall) if precision + recall else 0.0)
    return float(np.mean(scores)) if scores else 0.0

def bootstrap_ci(truth, prediction, metric=macro_f1, n_boot=1000, alpha=0.05, seed=SEED):
    truth = np.asarray(truth, dtype=object)
    prediction = np.asarray(prediction, dtype=object)
    if len(truth) == 0 or len(truth) != len(prediction):
        raise ValueError("paired non-empty arrays are required")
    generator = np.random.default_rng(seed)
    values = [
        metric(truth[idx], prediction[idx])
        for idx in (generator.integers(0, len(truth), len(truth)) for _ in range(n_boot))
    ]
    low, high = np.percentile(values, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return {"estimate": metric(truth, prediction),
            "ci_low": float(low), "ci_high": float(high), "n_boot": n_boot}

truth = [r["topic"] for r in validation_rows]
interval = bootstrap_ci(truth, list(baseline_predictions))
print("Baseline with interval [MEASURED_SMOKE]:", json.dumps(interval, indent=2))
assert interval["ci_low"] <= interval["estimate"] <= interval["ci_high"]
print("CONFIDENCE_INTERVAL=PASS")

In [ ]:
MIN_SLICE_SIZE = 15

def sliced_report(items, truth_key, prediction_lookup, keys):
    report = []
    for key in keys:
        for value in sorted({r[key] for r in items}):
            group = [r for r in items if r[key] == value]
            if not group:
                continue
            report.append({
                "slice": f"{key}={value}",
                "n": len(group),
                "flag": "SMALL_SLICE" if len(group) < MIN_SLICE_SIZE else "",
                "estimate": macro_f1([r[truth_key] for r in group],
                                     [prediction_lookup[r["example_id"]] for r in group]),
            })
    return report

lookup = {r["example_id"]: p for r, p in zip(validation_rows, baseline_predictions)}
slices = sliced_report(validation_rows, "topic", lookup, ["language"])
for row in slices:
    print("SLICE", row)

# Small slices are flagged, never dropped — deleting them hides the weakest cases.
assert any(row["flag"] == "SMALL_SLICE" for row in slices) or len(validation_rows) >= MIN_SLICE_SIZE
print("SLICED_EVALUATION=PASS")

## تحليل الأخطاء والإصلاحات المرتّبة | Error analysis and ranked fixes

الإصلاحات مرتّبة بحجم الأثر المقاس، لا بسهولة التنفيذ.

In [ ]:
RANKED_FIXES = [
    {"priority": 1, "issue": "Long inputs underperform short ones",
     "evidence": "length_bucket=long 0.526 vs short 0.829 (n=18 each, neither flagged SMALL_SLICE)",
     "action": "Audit length distribution against max_length; measure truncation on frozen data; evaluate sentence-aware chunking",
     "blocked_by": "sentencizer splits after the abbreviation 'د.'"},
    {"priority": 2, "issue": "Gulf dialect trails MSA",
     "evidence": "variant=Gulf 0.658 vs MSA 0.837; multilingual encoder scored 0.0 on the frozen Gulf test vs CAMeLBERT 0.6667",
     "action": "Run the Arabic-vs-multilingual comparison on frozen data with confidence intervals, on both language slices",
     "blocked_by": "bilingual scope — an Arabic-only encoder trades measured Arabic gain for unmeasured English loss"},
    {"priority": 3, "issue": "Arabic retrieval ranking is weaker than English",
     "evidence": "mrr@3 ar 0.5 vs en 0.833; recall@3 already 1.0, so the gap is ordering not retrieval",
     "action": "Re-measure on the frozen set with more queries before reopening the re-ranking decision",
     "blocked_by": "six answerable test queries is too few to separate signal from noise"},
]
for fix in RANKED_FIXES:
    print(f"[{fix['priority']}] {fix['issue']}\n    evidence: {fix['evidence']}\n    action:   {fix['action']}\n")
assert [f["priority"] for f in RANKED_FIXES] == [1, 2, 3]
print("RANKED_FIXES=PASS")

---
# 7) التحسين والخدمة | Optimisation and serving

**قِس قبل أن تُحسّن.** بلا خط أساس، «التحسين» ادعاء لا قياس.

**التكافؤ شرطان معًا:** فرق عددي صغير **و** اتفاق كامل في التنبؤات. الأول وحده لا يكفي — فرق ضئيل قد يقلب تنبؤًا عند الحدود.

In [ ]:
def percentiles(durations_ms):
    array = np.asarray(durations_ms, dtype=float)
    return {"p50_ms": float(np.percentile(array, 50)),
            "p95_ms": float(np.percentile(array, 95)),
            "p99_ms": float(np.percentile(array, 99))}

WARMUP, REPETITIONS = 5, 30
assert WARMUP >= 1 and REPETITIONS >= 30, "tail latency needs enough samples"

# Measured in notebooks/08_optimization_serving.ipynb with PROJECT_MODE=True.
BENCHMARK = {
    "result_label": "MEASURED",
    "artefact_role": "PROJECT_ARTIFACT",
    "device": "cpu / CPUExecutionProvider",
    "warmup": WARMUP,
    "repetitions": REPETITIONS,
    "budget_provenance": "STUDENT_DEFINED_BEFORE_MEASUREMENT",
    "workload_sha256": "9b07010d0e607282b66a0a1b1096175fd6015a237577db26bf03366393e077d8",
    "pytorch_fp32_p95_ms": 1107.054,
    "onnx_fp32_p95_ms": 137.120,
    "quality_tax": 0.0,
    "int8_available": True,
    "selected_for_service": "onnx-fp32",
    "adoption_decision": "ADOPT_ONNX_FP32",
}
speedup = BENCHMARK["pytorch_fp32_p95_ms"] / BENCHMARK["onnx_fp32_p95_ms"]
print(json.dumps(BENCHMARK, indent=2))
print(f"p95 speed-up: {speedup:.2f}x at a quality tax of {BENCHMARK['quality_tax']}")
assert BENCHMARK["quality_tax"] <= 0.05, "adoption requires the quality tax to stay within budget"
print("BENCHMARK_LADDER=PASS")

In [ ]:
def validate_request_text(text, max_chars=1000):
    if not isinstance(text, str) or not text.strip():
        raise ValueError("text must be a non-empty string")
    if len(text) > max_chars:
        raise ValueError(f"text exceeds {max_chars} characters")
    return text.strip()

def detect_language(text):
    arabic = sum("\u0600" <= ch <= "\u06ff" for ch in text)
    return "ar" if arabic else "en"

SERVING_MANIFEST = {
    "model_id": TOKENIZER_ID,
    "model_version": "project-v1",
    "preprocessing_version": f"{ACTIVE_PROFILE}-{PROFILE_VERSION}",
    "runtime": BENCHMARK["selected_for_service"],
    "labels": LABELS,
}

def service_predict(text):
    clean = validate_request_text(text)
    language = detect_language(clean)
    prepared = prepare_text(clean, ACTIVE_PROFILE)
    label = baseline.predict([prepared.model_text])[0]
    return {"language": language, "label": label,
            "preprocessing_version": SERVING_MANIFEST["preprocessing_version"]}

# Startup canaries: two contract cases, one per language.
CANARIES = [
    {"name": "arabic-contract",  "text": "الخدمة واضحة"},
    {"name": "english-contract", "text": "The service is clear"},
]
for canary in CANARIES:
    canary["expected"] = service_predict(canary["text"])
    print("CANARY", canary["name"], canary["expected"])
assert len(CANARIES) == 2

# The contract must reject bad input rather than guess.
contract = {}
contract["ar_valid"] = 200 if service_predict("تعذر تسجيل الدخول") else 500
contract["en_valid"] = 200 if service_predict("I cannot sign in") else 500
for name, bad in (("empty", ""), ("too_long", "x" * 2000)):
    try:
        service_predict(bad); contract[name] = 200
    except ValueError:
        contract[name] = 422

print("SERVICE_CONTRACT", contract)
assert contract["empty"] == 422 and contract["too_long"] == 422
assert all(code in {200, 422} for code in contract.values())
print("SERVICE_CONTRACT=PASS")

---
# 8) الملخص | Summary

كل رقم مع وسمه ومصدره. لا رقم بلا وسم، ولا ادعاء بلا دليل.

In [ ]:
SUMMARY = {
    "day1_preprocessing_tokenisation": {
        "label": "MEASURED_SMOKE",
        "source": "notebooks/01_text_processing_tokenization.ipynb",
        "arabic_fertility": 1.39, "english_fertility": 1.32,
        "truncation_at_8": 0.40, "truncation_at_10": 0.0, "truncation_at_16": 0.0,
    },
    "day2_classification": {
        "label": "MEASURED_SMOKE",
        "source": "notebooks/03_text_classification.ipynb",
        "baseline_macro_f1": 0.6667, "transformer_macro_f1": 0.55, "delta": -0.1167,
        "group_overlap": 0,
    },
    "day2_ner_qa": {
        "label": "MEASURED_SMOKE",
        "source": "notebooks/04_ner_and_qa.ipynb",
        "entity_f1": 0.0, "true_entities": 4, "predicted_entities": 0,
        "qa_valid_span": "الرياض", "qa_no_answer_returns_none": True,
    },
    "day3_arabic": {
        "label": "MEASURED_SMOKE",
        "source": "notebooks/05_arabic_nlp.ipynb",
        "multilingual_gulf_macro_f1": 0.0, "camelbert_gulf_macro_f1": 0.6667,
        "profile_version": PROFILE_VERSION,
    },
    "day3_search": {
        "label": "MEASURED_SMOKE",
        "source": "notebooks/06_semantic_search.ipynb",
        "recall_at_3": 1.0, "mrr_at_3": 0.6667,
        "mrr_ar": 0.5, "mrr_en": 0.8333,
        "frozen_threshold": FROZEN_THRESHOLD,
        "reranking": "REJECT_NO_MEASURED_LIFT",
    },
    "day3_evaluation": {
        "label": "COURSE_FIXTURE",
        "source": "notebooks/07_evaluation_error_analysis.ipynb",
        "fixture_a": {"estimate": 0.7807, "ci": [0.6212, 0.8982]},
        "fixture_b": {"estimate": 0.7819, "ci": [0.6169, 0.9042]},
        "intervals_overlap": True,
        "largest_gap": {"slice": "length_bucket", "long": 0.526, "short": 0.829},
    },
    "day4_serving": {
        "label": "MEASURED",
        "artefact_role": "PROJECT_ARTIFACT",
        "source": "notebooks/08_optimization_serving.ipynb",
        **{k: BENCHMARK[k] for k in
           ["pytorch_fp32_p95_ms", "onnx_fp32_p95_ms", "quality_tax",
            "int8_available", "adoption_decision"]},
    },
}

Path("reports/capstone_summary.json").write_text(
    json.dumps(SUMMARY, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(SUMMARY, ensure_ascii=False, indent=2))

## القيود المعروفة | Known limitations

1. **تقسيم الجمل يفشل عند الاختصارات.** `"راجع د. أحمد. ثم أعد المحاولة."` يعيد ثلاث جمل بدل اثنتين. موثّق لا مُخفى، ويحجب الإصلاح رقم 1.
2. **أغلب الأرقام `MEASURED_SMOKE` أو `COURSE_FIXTURE`.** قسم الخدمة وحده يحمل `PROJECT_ARTIFACT`.
3. **أحجام العينات 4 إلى 40 صفًا.** القسم 6 يبيّن أن الفترات على هذه البيانات تبتلع فروقًا أكبر من أغلب ما ورد.
4. **بذرة واحدة.** تباين البذور غير مقاس، فلم يُفصل أي فرق عن تقلب التشغيل.
5. **`p95 = 137 ms` على Colab CPU بحمل من 8 صفوف** — الهدف الرسمي `p99 ≤ 40 ms عند 16 concurrent` يُقاس في بيئة القياس المعلنة، لا هنا.
6. **إخفاء PII تعليمي.** أنماط البريد والجوال السعودي فقط، بلا اختبارات تغطية.
7. **متجهات البحث في هذا الدفتر توضيحية حتمية** لإبقائه قابلًا للتشغيل بلا اتصال؛ الأرقام المقاسة مصدرها الدفتر `06`.

In [ ]:
gate = {
    "setup": True,
    "split_isolation": split_report["group_overlap"] == 0,
    "golden_tests": True,
    "two_copy_contract": all(r.display_text == w["text"] for r, w in zip(records, rows)),
    "tokenisation_metrics": ar_fertility > 0 and en_fertility > 0,
    "baseline_measured": 0.0 <= baseline_f1 <= 1.0,
    "ner_alignment": aligned[0] == -100,
    "strict_entity_scoring": boundary["f1"] == 0.0,
    "qa_honest_null": unanswerable["answer"] is None,
    "search_l2_normalised": manifest["l2_normalised_both_sides"],
    "threshold_validation_only": manifest["threshold_tuned_on"] == "validation_only",
    "confidence_interval": interval["ci_low"] <= interval["estimate"] <= interval["ci_high"],
    "slices_flagged_not_dropped": len(slices) > 0,
    "ranked_fixes": [f["priority"] for f in RANKED_FIXES] == [1, 2, 3],
    "benchmark_within_budget": BENCHMARK["quality_tax"] <= 0.05,
    "service_contract": all(code in {200, 422} for code in contract.values()),
    "startup_canaries": len(CANARIES) == 2,
    "every_number_labelled": all("label" in section for section in SUMMARY.values()),
}
for name, passed in gate.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")
assert all(gate.values()), gate
print()
print("BAYAN_CAPSTONE_PIPELINE=PASS")

---

## الاعتماد | Acknowledgement

نُفذ هذا المشروع ضمن برنامج **SDA-AIE-211 — Natural Language Processing with Transformers**، بإعداد وتقديم المدربة **ميعاد المري**، ضمن برامج **أكاديمية سدايا**.

This project was completed as part of **SDA-AIE-211 — Natural Language Processing with Transformers**, prepared and delivered by instructor **Meaad Al-Marri**, within the programmes of **SDAIA Academy**.

**أكاديمية سدايا على GitHub:** https://github.com/SDAIAAcademy